# BHDAC Case Competition: Credit Risk Data Setup

**Goal of this notebook:** load the 300,000-account dataset, replace the coded column names (Acct1, BK1, Del3, ...) with names that say what each field actually measures, and save a renamed copy for the modeling work.

**Files needed in the same folder as this notebook** (both are in the shared drive folder *Case Competition Dataset & Description*):
- `U_of_DE_Comp.csv` (the data)
- `DataDictionary.xlsx` (columns: Column Name, Field Type, Description)

## 1. Imports and file paths

In [15]:
import pandas as pd

DATA_FILE = "U_of_DE_Comp.csv"
DICT_FILE = "DataDictionary.xlsx"

## 2. Load the data

`encoding="utf-8-sig"` removes a hidden byte-order mark at the very start of the CSV. Without it, the first column loads as `"\ufeffAcct1"` instead of `"Acct1"` and won't match the dictionary.

In [16]:
df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")
df.columns = df.columns.str.strip()

print(df.shape)
df.head()

(300000, 46)


,Acct1,Acct2,Acct3,Acct4,Acct5,Acct6,Acct7,Acct8,Acct9,Acct10,...,line_12m,line_24m,PubRec1,Repo1,Score1,Score2,Util1,Util2,Util3,Util4
0,83,1,2,0,204,1,89,0,1,1,...,1000,1000,0,0,715,689,40,60,40,13
1,83,3,19,0,205,17,96,0,0,0,...,19500,19500,0,0,654,668,73,56,89,9
2,84,9,15,2,123,12,68,0,0,0,...,1000,1000,0,0,627,626,75,76,81,12
3,84,9,16,0,204,25,82,0,0,2,...,11000,11000,0,0,730,722,34,93,17,11
4,83,1,4,0,196,1,100,0,0,2,...,10000,10000,0,0,794,808,5,8,6,8


## 3. Load the data dictionary

Each row in the dictionary describes one coded column. **Field Type** is one of:
- **Credit Attribute**: real predictors
- **Noise**: fields the case flags as not useful (deceased flag, natural disaster trades, etc.)
- **Performance**: outcome columns (the default flags, balances, and credit lines)

In [21]:
dd = pd.read_excel(DICT_FILE, usecols=["Column Name", "Field Type", "Description"])
dd["Column Name"] = dd["Column Name"].str.strip()
dd["Field Type"] = dd["Field Type"].str.strip()

print(dd["Field Type"].value_counts())
dd

Field Type
Credit Attribute    30
Noise               10
Performance          6
Name: count, dtype: int64


,Column Name,Field Type,Description
0,Acct1,Credit Attribute,Longest consecutive months with a satisfactory...
1,Acct2,Credit Attribute,Total number of revolving bankcard trades with...
2,Acct3,Credit Attribute,Total number of open trades
3,Acct4,Credit Attribute,Total number of trades opened in the last 3 mo...
4,Acct5,Credit Attribute,Number of months since the oldest trade was op...
5,Acct6,Credit Attribute,Total number of trades with a balance > $0 exc...
6,Acct7,Credit Attribute,Percentage of trades including collections tha...
7,BK1,Credit Attribute,Total number of bankruptcy trades
8,BK2,Credit Attribute,Total number of public record chapter 7 bankru...
9,BK3,Credit Attribute,Total number of public record chapter 13 bankr...


In [22]:
dd.columns

Index(['Column Name', 'Field Type', 'Description'], dtype='str')

## 4. Rename map: coded name to real meaning

Each new name is built from the dictionary description:
- `num_` = a count of trades or records
- `months_` = a number of months
- `pct_` / `util_` = a percentage or utilization ratio
- `bal_` = a dollar balance
- `dpd` = days past due

Acct12, Acct14, Score1, and Score2 have **no description** in the dictionary, so they get neutral names. Acct12 holds letter codes (BC, BB, FC, NU) and Acct14 looks like an ID, so check both before using them.

The Performance columns (`bad_flg_90plus_24m`, `bal_12m`, etc.) already have readable names and are the names the case description uses, so they are left as they are.

In [24]:
rename_map = {
    # Credit Attributes: account history
    "Acct1": "months_longest_satisfactory_streak",
    "Acct2": "num_active_revolving_bankcards_3m",
    "Acct3": "num_open_trades",
    "Acct4": "num_trades_opened_3m",
    "Acct5": "months_since_oldest_trade",
    "Acct6": "num_trades_with_balance",
    "Acct7": "pct_trades_never_delinquent",

    # Credit Attributes: bankruptcy
    "BK1": "num_bankruptcy_trades",
    "BK2": "num_ch7_bankruptcies",
    "BK3": "num_ch13_bankruptcies",

    # Credit Attributes: charge-offs and collections
    "CO1": "num_trades_ever_charged_off",
    "CO2": "num_unsatisfied_chargeoffs",
    "CO3": "bal_unsatisfied_chargeoffs",
    "Col1": "num_unsatisfied_collections",

    # Credit Attributes: delinquency
    "Del1": "num_trades_ever_30plus_dpd",
    "Del2": "num_trades_now_30dpd_6m",
    "Del3": "num_trades_now_90_180dpd_6m",
    "Del4": "num_trades_30plus_dpd_12m",
    "Del5": "num_trades_90dpd_24m",
    "Del6": "months_since_recent_30dpd",
    "Del8": "num_first_payment_defaults_12m",
    "Del9": "num_30dpd_occurrences_24m",

    # Credit Attributes: inquiries, public records, repossessions
    "Inq1": "num_inquiries_12m",
    "PubRec1": "num_judgments_over_1000",
    "Repo1": "num_trades_repossessed",

    # Credit Attributes: scores (no description given)
    "Score1": "credit_score_1",
    "Score2": "credit_score_2",

    # Credit Attributes: utilization
    "Util1": "util_overall_open_trades",
    "Util2": "util_highest_balance_trade_3m",
    "Util3": "util_revolving_bankcards_3m",

    # Noise
    "Acct8": "deceased_flag",
    "Acct9": "num_trades_natural_disaster",
    "Acct10": "num_no_preset_limit_bankcards",
    "Acct11": "months_since_recent_business_trade",
    "Acct12": "acct12_code_undocumented",
    "Acct13": "months_since_recent_personal_finance_trade",
    "Acct14": "acct14_undocumented",
    "Acct15": "num_mortgage_inquiries",
    "BK5": "num_discharged_bankruptcies",
    "Util4": "months_active_revolving_bankcards_12m",
}

print(len(rename_map), "columns in the rename map")

40 columns in the rename map


In [25]:
dd.columns

Index(['Column Name', 'Field Type', 'Description'], dtype='str')

## 5. Check the map against the data, then rename

Both lists are built **before** renaming, by comparing the data's column names to the keys of `rename_map` (the coded names):
- `renamed` = coded columns found in the data
- `unmatched` = data columns with no entry in the map (should be only the 6 Performance columns)
- `missing` = map entries that aren't in the data (should be empty; anything here means a typo)

In [26]:
df1 = df.copy()

renamed = [c for c in df1.columns if c in rename_map]
unmatched = [c for c in df1.columns if c not in rename_map]
missing = [c for c in rename_map if c not in df1.columns]

df1 = df1.rename(columns=rename_map)

print(f"Renamed {len(renamed)} fields")
print("Columns with no rename (kept as is):", unmatched)
print("Map entries not found in data:", missing)

Renamed 40 fields
Columns with no rename (kept as is): ['bad_flg_90plus_24m', 'bal_12m', 'bal_24m', 'consumer_bad_flg_90plus_24m', 'line_12m', 'line_24m']
Map entries not found in data: []


## 6. Lookup table: old name, new name, type, description

Keep this handy when reading model output or writing the deck. `.map()` looks up each coded name in `rename_map`; Performance columns aren't in the map, so `.fillna()` keeps their original name.

In [27]:
lookup = dd.copy()
lookup["New Name"] = lookup["Column Name"].map(rename_map).fillna(lookup["Column Name"])
lookup = lookup[["Column Name", "New Name", "Field Type", "Description"]]
lookup = lookup.sort_values(["Field Type", "Column Name"]).reset_index(drop=True)

pd.set_option("display.max_colwidth", 120)
lookup

,Column Name,New Name,Field Type,Description
0,Acct1,months_longest_satisfactory_streak,Credit Attribute,"Longest consecutive months with a satisfactory balance activity status on trades, excluding collections and other un..."
1,Acct2,num_active_revolving_bankcards_3m,Credit Attribute,Total number of revolving bankcard trades with a balance activity in the last 3 months excluding unsatisfied derogatory
2,Acct3,num_open_trades,Credit Attribute,Total number of open trades
3,Acct4,num_trades_opened_3m,Credit Attribute,Total number of trades opened in the last 3 months excluding collections including indeterminates
4,Acct5,months_since_oldest_trade,Credit Attribute,Number of months since the oldest trade was opened excluding collections including indeterminates
5,Acct6,num_trades_with_balance,Credit Attribute,Total number of trades with a balance > $0 excluding collections including indeterminates
6,Acct7,pct_trades_never_delinquent,Credit Attribute,Percentage of trades including collections that are never delinquent or derogatory
7,BK1,num_bankruptcy_trades,Credit Attribute,Total number of bankruptcy trades
8,BK2,num_ch7_bankruptcies,Credit Attribute,Total number of public record chapter 7 bankruptcies
9,BK3,num_ch13_bankruptcies,Credit Attribute,Total number of public record chapter 13 bankruptcies


## 7. Column lists by Field Type

Lists of the **new** names, so later cells can select groups, e.g. `df1[credit_cols]`.

In [28]:
credit_cols = list(lookup.loc[lookup["Field Type"] == "Credit Attribute", "New Name"])
noise_cols = list(lookup.loc[lookup["Field Type"] == "Noise", "New Name"])
performance_cols = list(lookup.loc[lookup["Field Type"] == "Performance", "New Name"])

print(len(credit_cols), "credit attributes")
print(len(noise_cols), "noise fields")
print(len(performance_cols), "performance fields")

30 credit attributes
10 noise fields
6 performance fields


## 8. Sanity check against the case description

The case says the baseline default rates are **6.31%** for `bad_flg_90plus_24m` (this account) and **9.12%** for `consumer_bad_flg_90plus_24m` (any account). If these match, the data loaded correctly.

In [29]:
account_rate = df1["bad_flg_90plus_24m"].mean() * 100
consumer_rate = df1["consumer_bad_flg_90plus_24m"].mean() * 100

print(f"Account default rate:  {account_rate:.2f}%  (expected 6.31%)")
print(f"Consumer default rate: {consumer_rate:.2f}%  (expected 9.12%)")

Account default rate:  6.31%  (expected 6.31%)
Consumer default rate: 9.12%  (expected 9.12%)


## 9. Save the renamed data and the lookup table

In [30]:
df1.to_csv("U_of_DE_Comp_renamed.csv", index=False)
lookup.to_csv("column_lookup.csv", index=False)

print("Saved U_of_DE_Comp_renamed.csv and column_lookup.csv")

Saved U_of_DE_Comp_renamed.csv and column_lookup.csv
